# 01 · Data and network structure

Builds the analysis dataset from the harmonised dMRI connectomes and draws the structural
figures of the manuscript.

| | |
|---|---|
| **inputs** | `preprocdata/all_data.mat` (connectomes), `preprocdata/demographics.csv`, the AAL atlas `preprocdata/aal-SPM12/aal/atlas/AAL.nii`, the Yeo-7 atlas `preprocdata/Yeo2011_7Networks_MNI152_FreeSurferConformed1mm_LiberalMask.nii.gz` |
| **outputs** | `procdata/data_with_metrics.pkl` (one row per connectome; used by notebooks 02 and 03), `procdata/grep_ageds.pkl`, `procdata/yeo7_aal90_mapping.pkl`, `procdata/fig2_precomp.pkl` |
| **figures** | `FIG_DIR/`: `figdata_atlas_age.pdf` (Fig. 1), `fig1_distributions.pdf`, `fig2_grid.pdf`, `fig2_boxplots.pdf`, `figSI_Y_horserace.pdf` |

Run order of the repository: **01 → 02 → 03**. The expensive extra metrics are cached in
`procdata/data_with_metrics.pkl`; delete it to recompute them.

In [ ]:
# This notebook works from the repository root (data in preprocdata/ and procdata/).
import os
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "engine" / "esn.hpp").exists())
os.chdir(ROOT)
print("working directory:", Path.cwd())

## 0 · Imports, style and parameters

In [ ]:
import os, gzip, shutil, pickle, warnings
from collections import defaultdict, Counter
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import pandas as pd
import networkx as nx
import nibabel as nib
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.cm as cm
import matplotlib.gridspec as gridspec
from matplotlib.patches import Patch
import seaborn as sns
from joblib import Parallel, delayed
from nilearn import plotting, datasets, image
from pymatreader import read_mat
from scipy.linalg import expm, eigvals
from scipy.spatial.distance import cdist
from scipy.stats import pearsonr, linregress
from statsmodels.stats.multitest import multipletests

warnings.filterwarnings('ignore')
plt.rcParams.update({
    'font.size':7,'axes.titlesize':7,'axes.labelsize':7,
    'xtick.labelsize':8,'ytick.labelsize':8,
    'lines.linewidth':0.9,'lines.markersize':2.5,'figure.dpi':300,
})
sns.set_theme(style='ticks', context='paper')

In [ ]:
# ---- data and outputs ----------------------------------------------------------------
CONNECTOMES_MAT = 'preprocdata/all_data.mat'
DEMOGRAPHICS    = 'preprocdata/demographics.csv'
AAL_NII         = './preprocdata/aal-SPM12/aal/atlas/AAL.nii'
FIG_DIR         = 'borradorv2'          # figures of this notebook
os.makedirs('procdata', exist_ok=True); os.makedirs(FIG_DIR, exist_ok=True)

# ---- sample and analysis constants ---------------------------------------------------
FILTER_DS = [1,2,3,4,5,6,7,8,9]         # datasets kept (neurotypical subjects only)
DS_NAMES  = {1: 'dHCP', 2: 'BCP', 3: 'CALM', 4: 'RED', 5: 'ACE',
             6: 'HCPd', 7: 'HCPya', 8: 'camCAN', 9: 'HCPa'}
AGE_SPLIT = 32                          # development (age <= 32) vs aging (age >= 32)
N_BINS    = 22                          # distance bins of the DDW group representatives
MIN_N     = 3                           # minimum subjects per group / decade
N_WORKERS = -1                          # joblib workers (-1 = all cores)

## 1 · Helper functions

**Subject-level metrics.** For each node: degree $k_i$, strength $s_i$, disparity
$Y_i=\sum_j (w_{ij}/s_i)^2$ and $\Upsilon_i = k_iY_i$. Per subject: density, mean strength and
degree, mean $Y$, mean $kY$, the null expectation $Y_{null}=2/(k+1)$ (random partition of the
strength) and the ratio $R = \langle Y/Y_{null}\rangle$.

In [ ]:
def null_model_kY(ks=None):
    """
    Reference curves for the kY(k) scatter:
      kY_null (mean), kY_hi (mean+2σ), kY_lo (=1),
      kY_hom (fully homogeneous), kY_het (fully heterogeneous)
    """
    if ks is None:
        ks = np.arange(2, 90)
    kY_null = 2 * ks / (ks + 1)
    term1   = (20 + 4*ks) / ((ks+1) * (ks+2) * (ks+3))
    term2   = 4 / (ks+1)**2
    var_kY  = (ks**2) * (term1 - term2)
    return dict(
        ks      = ks,
        kY_null = kY_null,
        kY_hi   = kY_null + 2*np.sqrt(var_kY),
        kY_lo   = np.ones_like(ks, dtype=float),
        kY_hom  = np.ones_like(ks, dtype=float),
        kY_het  = ks.astype(float),
    )


def make_ds_colors(ds_list):
    """dict {dataset_id: colour} from tab10."""
    cmap = plt.cm.tab10
    return {ds: cmap(i / max(len(ds_list) - 1, 1)) for i, ds in enumerate(ds_list)}


def node_metrics(adj):
    """
    Node metrics of a weighted adjacency matrix.
    Returns: k (degree), s (strength), Y (disparity), kY (k·Y), w (mean weight)
    """
    A = adj.copy(); np.fill_diagonal(A, 0)
    k  = (A > 0).sum(axis=1).astype(float)
    s  = A.sum(axis=1)
    p  = A / np.where(s > 0, s, 1.)[:, None]
    Yi = (p**2).sum(axis=1)
    # average weight of a node
    wi = np.divide(s, k, out=np.zeros_like(s, dtype=float), where=k>0)
    return k, s, Yi, k * Yi, wi


def subject_metrics(conn_array, N_roi=90):
    """
    Scalar metrics per subject for an array of connectomes.
    Returns a DataFrame with: density, strength, degree, Y_obs, kY_obs, Y_null, Ratio
    """
    n = len(conn_array)
    D, S, K, Y, kY, YN, R = [np.zeros(n) for _ in range(7)]
    for i, raw in enumerate(conn_array):
        adj = raw.copy(); np.fill_diagonal(adj, 0)
        k, s, Yi, kYi, wi = node_metrics(adj)
        v = k > 1
        if v.sum() == 0:
            D[i] = S[i] = K[i] = Y[i] = kY[i] = YN[i] = R[i] = np.nan
            continue
        D[i]  = (adj > 0).sum() / (N_roi * (N_roi - 1))
        S[i]  = s[v].mean()
        K[i]  = k[v].mean()
        Y[i]  = Yi[v].mean()
        kY[i] = kYi[v].mean()
        YN[i] = (2. / (k[v] + 1)).mean()
        R[i]  = (Yi[v] / (2. / (k[v] + 1))).mean()
    return pd.DataFrame(dict(density=D, strength=S, degree=K,
                             Y_obs=Y, kY_obs=kY, Y_null=YN, Ratio=R, ))

**Extra graph metrics** (cached in `data_with_metrics.pkl`): unweighted clustering (networkx),
weighted clustering (Onnela et al. 2005), and the global communicability and mean subgraph
centrality of $e^{\tilde W}$, with $\tilde W$ rescaled to spectral radius $\rho = 0.99$.

In [ ]:
def _onnela_clustering_global(W):
    """Onnela et al. 2005 — consistent with _process_subject."""
    w_max = W.max()
    if w_max == 0: return 0.0
    W_cbrt = np.cbrt(W / w_max)
    k      = (W > 0).sum(axis=1).astype(float)
    num    = np.sum((W_cbrt @ W_cbrt) * W_cbrt, axis=1)
    denom  = k * (k - 1)
    with np.errstate(invalid='ignore', divide='ignore'):
        c_i = np.where(denom > 0, num / denom, 0.0)
    return c_i.mean()


def _metrics_one_thread(args):
    idx, A, rho = args
    A = A.copy(); np.fill_diagonal(A, 0)
    N = A.shape[0]

    # Unweighted — networkx OK
    G_unw = nx.from_numpy_array((A > 0).astype(float))
    c_unw = nx.average_clustering(G_unw)

    # Weighted — Onnela, same as _process_subject
    c_w = _onnela_clustering_global(A)

    # Global communicability
    sr = np.abs(eigvals(A)).max()
    A_norm = A * (rho / sr) if sr > 0 else A
    try:
        C    = expm(A_norm)
        comm = (C.sum() - np.trace(C)) / (N * (N - 1))
        sc   = np.diag(C).mean()
    except Exception:
        comm = np.nan
        sc   = np.full(N, np.nan)

    return idx, c_unw, c_w, comm, sc


def compute_extra_metrics(df, rho=0.99, n_workers=8):
    df = df.copy()
    c_unw_out = [None] * len(df)
    c_w_out   = [None] * len(df)
    comm_out  = [None] * len(df)
    sc_out    = [None] * len(df)

    args_list = [
        (idx, np.asarray(row.connectome, dtype=np.float64), rho)
        for idx, (_, row) in enumerate(df.iterrows())
    ]

    with ThreadPoolExecutor(max_workers=n_workers) as executor:
        futures = {executor.submit(_metrics_one_thread, a): a[0] for a in args_list}
        for fut in as_completed(futures):
            idx, c_unw, c_w, comm, sc = fut.result()
            c_unw_out[idx] = c_unw
            c_w_out[idx]   = c_w
            comm_out[idx]  = comm
            sc_out[idx]    = sc

    df['C_unw']     = c_unw_out
    df['C_w']       = c_w_out
    df['comm_mean'] = comm_out
    df['sc_mean']   = sc_out
    return df

**Group representatives (DDW).** Distance-dependent consensus connectome of a group of
subjects: topology from Betzel's `fcn_group_bins.m` (via the Python port of Levakov,
GidLev/consensus-thresholding), weights sampled from the group as in Mišić et al. (2015).

In [ ]:
def ddw_group_rep(connectomes, dist_mat, hemi, n_bins=22, rng=None):
    """
    DDW group-representative connectome.

    Topology: faithful translation of Betzel's official code (fcn_group_bins.m),
    via the Python implementation of Levakov (GidLev/consensus-thresholding).
    Weights: random DDW sampling as in Mišić et al. (2015).

    Parameters
    ----------
    connectomes : list of np.ndarray (N, N)
    dist_mat    : np.ndarray (N, N)  — distances between ROIs
    hemi        : np.ndarray (N,)    — hemisphere of each node (0=left, 1=right)
    n_bins      : int                — number of distance bins (default 22)
    rng         : np.random.Generator | None

    Returns
    -------
    group_rep : np.ndarray (N, N)  — symmetric, weighted
    """
    if rng is None:
        rng = np.random.default_rng(42)

    connectomes = [np.asarray(c, dtype=float) for c in connectomes]
    N     = connectomes[0].shape[0]
    nsub  = len(connectomes)

    # 3D stack: (N, N, nsub)
    adj = np.stack(connectomes, axis=2)

    # Consistency and mean weight (exactly as Betzel)
    C = np.sum(adj > 0, axis=2)           # number of subjects with this edge
    with np.errstate(invalid='ignore'):
        W = np.sum(adj, axis=2) / C       # mean weight among those who have it
    W[np.isnan(W)] = 0

    # Weight pool for DDW sampling
    iu_r, iu_c = np.triu_indices(N, k=1)
    weights_pool = {}
    for r, c in zip(iu_r, iu_c):
        ws = adj[r, c, :]
        ws = ws[ws > 0]
        if len(ws) > 0:
            weights_pool[(r, c)] = ws

    # Linear distance bins over the non-zero distances
    # (as Betzel: np.linspace over [min_nonzero, max])
    nonzero_dists = dist_mat[dist_mat > 0]
    distbins = np.linspace(nonzero_dists.min(), nonzero_dists.max(), n_bins + 1)
    distbins[-1] += 1.0   # include the exact maximum

    # 2D inter-hemispheric mask (as Betzel)
    hemi_col = hemi[:, np.newaxis]
    inter_mask_2d = np.dot(hemi_col, (~hemi_col.astype(bool)).T.astype(int)).astype(bool)
    inter_mask_2d = inter_mask_2d | inter_mask_2d.T   # symmetrise

    Grp = np.zeros((N, N))   # accumulated topology (intra + inter)

    for j in range(2):
        # j=0 → inter-hemispheric, j=1 → intra-hemispheric  (as Betzel)
        if j == 0:
            hemi_mask = inter_mask_2d
        else:
            hemi_mask = ~inter_mask_2d

        m = dist_mat * hemi_mask   # distances of this group only

        # Distances of the edges present in at least one subject, in this group
        # D = distances of all present edges (upper triangle) * hemi_mask
        present_upper = np.triu(hemi_mask, 1) & (C > 0)
        D = dist_mat[present_upper]   # 1D: distances of the present edges

        if len(D) == 0:
            continue

        tgt = len(D) / nsub   # mean number of edges per subject in this group

        G = np.zeros((N, N))

        for i_bin in range(n_bins):
            lo, hi = distbins[i_bin], distbins[i_bin + 1]

            # Candidate pairs: upper triangle, this hemisphere group, this bin
            mask_ij = np.where(
                np.triu((m >= lo) & (m < hi), 1)
            )
            if len(mask_ij[0]) == 0:
                continue

            # frac = tgt * (fraction of the present edges that fall in this bin)
            # exactly as Betzel: tgt * sum(D in bin) / len(D)
            frac = int(np.round(tgt * np.sum((D >= lo) & (D < hi)) / len(D)))
            if frac == 0:
                continue

            # Sort by consistency (descending) and keep the top frac
            c_vals = C[mask_ij]
            idx    = np.argsort(c_vals)[::-1]
            chosen = min(frac, len(idx))

            G[mask_ij[0][idx[:chosen]], mask_ij[1][idx[:chosen]]] = 1

        Grp += G

    # Symmetrise the topology
    Grp = (Grp + Grp.T).astype(bool)

    # Assign weights by DDW sampling
    group_rep = np.zeros((N, N))
    for r, c in zip(*np.where(np.triu(Grp, 1))):
        key = (r, c)
        if key in weights_pool:
            w = rng.choice(weights_pool[key])
            group_rep[r, c] = w
            group_rep[c, r] = w

    return group_rep

## 2 · Load the data

Neurotypical subjects (`group == 1`) of the nine datasets, with their connectomes and the
scalar metrics of §1. The AAL centroids give the inter-node distances and hemispheres used by
the group representatives.

In [ ]:
connectomes = read_mat(CONNECTOMES_MAT)['all_data']['connectomes']
demo        = pd.read_csv(DEMOGRAPHICS)

mask = (demo['group']==1) & (demo['dataset'].isin(FILTER_DS))       # neurotypical subjects only
demo        = demo[mask].reset_index(drop=True)
connectomes = connectomes[mask]

data = pd.DataFrame({'age':demo['age'], 'dataset':demo['dataset'],
                     'sex':demo['sex'], 'connectome':list(connectomes)})
data['decade'] = (data.age // 10 * 10).astype(int)

# Scalar metrics
data = pd.concat([data.reset_index(drop=True),
                  subject_metrics(connectomes)], axis=1)

# Colours and globals
ds_list = sorted(data.dataset.unique())
ND      = len(ds_list)
ds2col  = make_ds_colors(ds_list)
NUL     = null_model_kY()    # dict with ks, kY_null, kY_hi, kY_lo, kY_hom, kY_het

print(f'{len(data)} subjects, {ND} datasets')

coords   = plotting.find_parcellation_cut_coords(AAL_NII)[:90]
dist_mat = cdist(coords, coords)
hemi = np.zeros(90, dtype=int)
hemi[:45] = 1
rng_ddw  = np.random.default_rng(42)

**Group representatives** per dataset and age (at least 3 subjects per group), saved to
`procdata/grep_ageds.pkl`.

In [ ]:
# Group representatives per dataset and age (DDW)
grep_ageds = pd.DataFrame()
for ds in ds_list:
    for age in sorted(data[data.dataset==ds].age.unique()):
        sub = data[(data.dataset==ds) & (data.age==age)]
        # at least 3 subjects for a meaningful group representative
        if len(sub) < 3: continue
        gr = ddw_group_rep(list(sub.connectome), dist_mat, hemi, N_BINS, rng_ddw)
        metrics_gr = subject_metrics([gr])  # metrics of the group representative
        # metadata of the current group (constant within 'sub')
        meta = pd.DataFrame({
            'age': [age],
            'dataset': [ds],
            'connectome': [gr]
        })
        # metrics and metadata side by side, appended to the table
        row = pd.concat([metrics_gr.reset_index(drop=True), meta], axis=1)
        grep_ageds = pd.concat([grep_ageds, row], ignore_index=True)

**Extra metrics and export.** Loads `procdata/data_with_metrics.pkl` if it exists; otherwise
computes the extra metrics of §1 and writes it. This pickle is the input of notebooks 02 and 03.

In [ ]:
try:
    with open('procdata/data_with_metrics.pkl', 'rb') as f:
        data = pickle.load(f)
    print(f"data loaded from disk ({len(data)} subjects).")
    print(f"Columns: {list(data.columns)}")
except FileNotFoundError:
    print("Computing C_unw, C_w, comm_mean (first run)...")
    data = compute_extra_metrics(data)
    with open('procdata/data_with_metrics.pkl', 'wb') as f:
        pickle.dump(data, f)
    print(f"Saved. Columns: {list(data.columns)}")

with open('procdata/grep_ageds.pkl', 'wb') as f:
    pickle.dump(grep_ageds, f)
print("grep_ageds saved.")

## 3 · Figure 1 and the distributions figure

`fig1_distributions.pdf`: per dataset, the weight and strength distributions, $s(k)$, the
degree CCDF, the (weighted) average nearest-neighbour degree and the (weighted) clustering
spectrum, with exponential binning on raw weights. `figdata_atlas_age.pdf` (Fig. 1 of the
manuscript): the AAL atlas and the age distribution per dataset.

In [ ]:
# ---- distribution accumulators -------------------------------------------------------
_BINS = {
    'w': np.logspace(1, 6, 100),
    's': np.logspace(0, 7, 200),
    'k': np.logspace(np.log10(2), np.log10(89), 100),
}

def _exp_histogram(values, bins):
    counts, edges = np.histogram(values[values > 0], bins=bins, density=True)
    centres = np.sqrt(edges[:-1] * edges[1:])
    ok = counts > 0
    return centres[ok], counts[ok]

def _exp_profile(x, y, bins):
    centres = np.sqrt(bins[:-1] * bins[1:])
    idx = np.digitize(x, bins) - 1
    return {c: np.mean(y[idx == i])
            for i, c in enumerate(centres) if np.any(idx == i)}

def accumulate_distributions(subjects) -> dict:
    """Single-pass accumulation — raw weights, no per-subject normalisation."""
    pw_acc = defaultdict(list); ps_acc = defaultdict(list)
    knn_uw_acc = defaultdict(list); knn_w_acc = defaultdict(list)
    ck_uw_acc  = defaultdict(list); ck_w_acc  = defaultdict(list)
    sk_acc = defaultdict(list);  cpk_acc = defaultdict(list)
    cpk_raw_acc = defaultdict(list)
    raw_k, raw_s = [], []
    k_eval_norm = np.logspace(-2, 0, 40)
    k_eval_raw = np.logspace(0, np.log10(89), 40)

    for row in subjects:
        adj = (row.connectome if hasattr(row, 'connectome') else row).copy()
        np.fill_diagonal(adj, 0)
        k, s, Yi, kYi, wi = node_metrics(adj)
        w     = adj[adj > 0]
        valid = k > 1
        if valid.sum() == 0:
            continue
        k_v, s_v = k[valid], s[valid]
        k_pos    = k[k > 0]
        k2_over_k = np.mean(k_pos**2) / np.mean(k_pos)

        A   = (adj > 0).astype(np.float64)
        A2  = A @ A
        knn_uw = (A   @ k) / np.where(k > 0, k.astype(float), 1.0)
        knn_w  = (adj @ k) / np.where(s > 0, s, 1.0)
        denom_k = k * (k - 1)
        C_uw    = np.where(denom_k > 0, (A2 * A).sum(axis=1)   / denom_k, 0.0)
        
        w_max   = adj.max()
        W_hat   = adj / w_max if w_max > 0 else adj
        W13     = np.cbrt(W_hat)                          # element-wise cube root (0→0)
        tri_o   = np.einsum('ij,jk,ki->i', W13, W13, W13) # = (W13³)_ii
        C_w = np.where(denom_k > 0, tri_o / denom_k, 0.0)
        raw_k.append(k_v); raw_s.append(s_v)
        for c, h in zip(*_exp_histogram(w,   _BINS['w'])): pw_acc[c].append(h)
        for c, h in zip(*_exp_histogram(s_v, _BINS['s'])): ps_acc[c].append(h)
        for c, v in _exp_profile(k_v, knn_uw[valid] / k2_over_k, _BINS['k']).items(): knn_uw_acc[c].append(v)
        for c, v in _exp_profile(k_v, knn_w[valid]  / k2_over_k, _BINS['k']).items(): knn_w_acc[c].append(v)
        mc_uw = C_uw[valid][C_uw[valid] > 0].mean() if (C_uw[valid] > 0).any() else 1.0
        mc_w  = C_w[valid][C_w[valid]   > 0].mean() if (C_w[valid]  > 0).any() else 1.0
        for c, v in _exp_profile(k_v, C_uw[valid] / mc_uw, _BINS['k']).items(): ck_uw_acc[c].append(v)
        for c, v in _exp_profile(k_v, C_w[valid]  / mc_w,  _BINS['k']).items(): ck_w_acc[c].append(v)
        ms = s_v.mean() if s_v.size > 0 else 1.0
        for c, v in _exp_profile(k_v, s_v / ms, _BINS['k']).items(): sk_acc[c].append(v)
        k_norm = k_v / k_v.max()
        for c in k_eval_norm:
            prob = np.mean(k_norm >= c)
            if prob > 0: cpk_acc[c].append(prob)
        for c in k_eval_raw:
            prob = np.mean(k_v >= c)
            if prob > 0: cpk_raw_acc[c].append(prob)

    return dict(pw=pw_acc, ps=ps_acc, knn_uw=knn_uw_acc, knn_w=knn_w_acc,
                ck_uw=ck_uw_acc, ck_w=ck_w_acc, sk=sk_acc, cpk=cpk_acc,
                cpk_raw=cpk_raw_acc,
                all_k=np.concatenate(raw_k) if raw_k else np.array([]),
                all_s=np.concatenate(raw_s) if raw_s else np.array([]))

In [ ]:
# ---- <w>_k per decade: exponential binning on raw integer k -----------------------
_BINS_WK = np.logspace(np.log10(2), np.log10(89), 100)   # k bins

def _wk_acc_expbin(sub_df) -> dict:
    """
    ⟨w⟩_k with EXPONENTIAL BINNING on raw integer k.
    FIX vs get_wk_stats(): no A/A.max() normalisation, k in raw integer space.
    """
    acc   = defaultdict(list)
    cents = np.sqrt(_BINS_WK[:-1] * _BINS_WK[1:])

    for _, row in sub_df.iterrows():
        A = row.connectome.copy()
        np.fill_diagonal(A, 0)
        if A.max() == 0:
            continue
        # ── KEY FIX: do NOT normalise A ──────────────────────────────────────
        k      = (A > 0).sum(axis=1).astype(float)
        mean_w = np.where(k > 0, A.sum(axis=1) / k, 0.0)
        valid  = k > 1

        idx = np.digitize(k[valid], _BINS_WK) - 1
        for i, c in enumerate(cents):
            m = idx == i
            if m.any():
                acc[c].append(mean_w[valid][m].mean())
    return acc


def _acc_to_curves(acc: dict):
    """Median + IQR from {x: [values]} accumulator."""
    xs  = np.array(sorted(acc))
    med = np.array([np.median(acc[x])         for x in xs])
    p25 = np.array([np.percentile(acc[x], 25) for x in xs])
    p75 = np.array([np.percentile(acc[x], 75) for x in xs])
    ok  = med > 0
    return xs[ok], med[ok], p25[ok], p75[ok]

In [ ]:
# ---- parallel pre-computation: one dataset / decade per job (joblib threads) --------

def _compute_ds(ds, data, ds2col):
    """Worker: accumulate distributions for one dataset."""
    subjects_iter = data[data.dataset == ds].itertuples()
    acc = accumulate_distributions(subjects_iter)
    return {'acc': acc, 'color': ds2col[ds], 'alpha': 0.8, 'lw': 1.2}

def _compute_decade(dec, data, cmap_wk, norm_wk):
    """Worker: compute w(k) curves for one decade."""
    sub_dec = data[data.decade == dec]
    acc = _wk_acc_expbin(sub_dec)          # ← fixed helper
    if not acc:
        return None
    ks, w_med, p25, p75 = _acc_to_curves(acc)
    return {'ks': ks, 'w_med': w_med, 'p25': p25, 'p75': p75,
            'color': cmap_wk(norm_wk(dec)), 'dec': dec}

def precompute_parallel(data, ds2col, FILTER_DS, n_workers=-1):
    """
    Notebook-safe parallel pre-computation using joblib.
    n_workers=-1 uses all available cores.
    """
    all_decades   = sorted(data['decade'].unique())
    valid_decades = [d for d in all_decades
                     if len(data[data.decade == d]) >= MIN_N and d <= 80]
    norm_wk = mcolors.Normalize(vmin=min(valid_decades), vmax=max(valid_decades))
    cmap_wk = plt.cm.plasma

    # ── Dataset distributions (embarrassingly parallel) ───────────────────────
    left_accumulated = Parallel(n_jobs=n_workers, prefer='threads')(
        delayed(_compute_ds)(ds, data, ds2col) for ds in FILTER_DS
    )

    # ── w(k) per decade ───────────────────────────────────────────────────────
    wk_results = Parallel(n_jobs=n_workers, prefer='threads')(
        delayed(_compute_decade)(dec, data, cmap_wk, norm_wk)
        for dec in valid_decades
    )

    wk_calculated = [r for r in wk_results if r is not None]
    # Sort by decade so colorbar is ordered
    wk_calculated.sort(key=lambda x: x['dec'])

    return left_accumulated, wk_calculated, valid_decades, norm_wk, cmap_wk

In [ ]:
# ---- distributions figure -----------------------------------------------------------

_PANEL_KEYS = ['pw', 'ps', 'sk', 'cpk_raw', 'knn_uw', 'knn_w', 'ck_uw', 'ck_w']
_XLABELS    = [r'$w$', r'$s$', r'$k$', r'$k$', r'$k$', r'$k$', r'$k$', r'$k$']
_YLABELS    = [
    r'$P(w)$', r'$P(s)$',
    r'$\langle s\rangle_k/\langle s\rangle$',
    r'$P(K \geq k)$',
    r'$\bar{k}_{nn}(k)/ \kappa$',
    r'$\bar{k}_{nn}^w(k)/\kappa $',
    r'$C(k)/\langle C\rangle$', r'$C^w(k)/\langle C^w\rangle$'
]

def build_distributions_fig(left_accumulated, ds2col, DS_NAMES, data, figsize=(3.4, 7.2)):
    fig = plt.figure(figsize=figsize)
    # 1. Use a strict 4x2 grid without the 5th row dummy axis
    gs = gridspec.GridSpec(4, 2, hspace=0.6, wspace=0.6)
    axs = [fig.add_subplot(gs[r, c]) for r in range(4) for c in range(2)]

    for item in left_accumulated:
        if item is None:
            continue
        acc, c, a, lw = item['acc'], item['color'], item['alpha'], item['lw']
        for ax, key in zip(axs, _PANEL_KEYS):
            if not acc.get(key):
                continue
            xs  = np.array(sorted(acc[key]))
            med = np.array([np.median(acc[key][x]) for x in xs])
            ok  = med > 0
            ax.loglog(xs[ok], med[ok], '-', lw=lw, alpha=a, color=c)

    ax_d = axs[3]
    axins = ax_d.inset_axes([0.2, 0.3, 0.42, 0.42])
    for item in left_accumulated:
        if item is None: continue
        acc = item['acc']
        if acc.get('cpk'):
            xs  = np.array(sorted(acc['cpk']))
            med = np.array([np.median(acc['cpk'][x]) for x in xs])
            ok  = med > 0
            axins.loglog(xs[ok], med[ok], '-', lw=0.8,
                     alpha=item['alpha'], color=item['color'])
    axins.set_xlabel(r'$k/k_{\max}$', fontsize=5, labelpad=0)
    axins.tick_params(labelsize=4, pad=1)
    sns.despine(ax=axins)
    
    for ax, xl, yl, lab in zip(axs, _XLABELS, _YLABELS, 'abcdefgh'):
        ax.set_xlabel(xl, fontsize=7); ax.set_ylabel(yl, fontsize=7, labelpad=1)
        ax.tick_params(labelsize=6); sns.despine(ax=ax)
        ax.set_title(f"{lab}", loc='left', fontsize=9, fontweight='bold')
    
    for ax, xl, yl in zip(axs, _XLABELS, _YLABELS):
        # linear y-axis for panels e-h
        for ax in axs[4:]:
            ax.set_yscale('linear')
            ax.yaxis.set_major_locator(plt.MaxNLocator(4))

    # 2. Add figure-level legend at the bottom
    handles = [Patch(facecolor=ds2col[ds], alpha=0.8, label=DS_NAMES[ds])
               for ds in sorted(DS_NAMES.keys()) if ds in data['dataset'].values]
               
    fig.subplots_adjust(bottom=0.15) # Allocate space at the bottom boundary
    fig.legend(handles=handles, loc='lower center', bbox_to_anchor=(0.5, 0.0), 
               ncol=3, frameon=True, fontsize=9, handlelength=1.0, 
               handletextpad=0.5, columnspacing=1.0)
               
    return fig

In [ ]:
def build_data_fig(data, atlas_nii_path, ds2col, DS_NAMES,
                   BINS_AGE=np.arange(0, 96, 2), figsize=(3.4, 1.9)):
    fig = plt.figure(figsize=figsize)
    gs  = gridspec.GridSpec(1, 2, width_ratios=[1, 1], wspace=0.35)

    # --- 1. Brain Atlas Panels (Vertically Stacked) ---
    # Subdivide the left column into 2 rows for X and Z modes
    gs_brain = gridspec.GridSpecFromSubplotSpec(2, 1, subplot_spec=gs[0, 0], hspace=0.05)
    ax_x = fig.add_subplot(gs_brain[0, 0])
    ax_z = fig.add_subplot(gs_brain[1, 0])
    
    mni = datasets.fetch_icbm152_2009().t1
    # Plot X (Sagittal) - force 1 slice
    plotting.plot_roi(atlas_nii_path, bg_img=mni, display_mode='x', cut_coords=1, 
                      alpha=0.6, cmap='tab20', annotate=False, draw_cross=False,
                      black_bg=False, colorbar=False, axes=ax_x)
    ax_x.set_title('a', fontsize=9, loc='left', fontweight='bold')

    # Plot Z (Axial) - force 1 slice
    plotting.plot_roi(atlas_nii_path, bg_img=mni, display_mode='z', cut_coords=1, 
                      alpha=0.6, cmap='tab20', annotate=False, draw_cross=False,
                      black_bg=False, colorbar=False, axes=ax_z)

    # --- 2. Age Histogram Panel ---
    ax_hist = fig.add_subplot(gs[0, 1])
    bottoms = np.zeros(len(BINS_AGE) - 1)
    for ds in sorted(DS_NAMES.keys()):
        if ds not in data['dataset'].values:
            continue
        hist, _ = np.histogram(data[data.dataset == ds]['age'].values, bins=BINS_AGE)
        ax_hist.bar(BINS_AGE[:-1], hist, width=np.diff(BINS_AGE), bottom=bottoms,
                    color=ds2col[ds], alpha=0.6, align='edge', linewidth=0)
        bottoms += hist
        
    ax_hist.set_xlabel('Age', fontsize=7); ax_hist.set_ylabel('Frequency', fontsize=7)
    ax_hist.set_xticks(np.arange(0, 100, 20)); ax_hist.tick_params(labelsize=6)
    sns.despine(ax=ax_hist); ax_hist.set_title('b', fontsize=9, loc='left', fontweight='bold')
    
    handles = [Patch(facecolor=ds2col[ds], alpha=0.8, label=DS_NAMES[ds])
               for ds in sorted(DS_NAMES.keys()) if ds in data['dataset'].values]
    ax_hist.legend(handles=handles, loc='upper right', ncol=2, frameon=True,
                   fontsize=5, handlelength=0.8, handletextpad=0.4, columnspacing=0.8)
                   
    return fig

In [ ]:
left_accumulated, wk_calculated, valid_decades, norm_wk, cmap_wk = precompute_parallel(
    data, ds2col, FILTER_DS, n_workers=8)
build_distributions_fig(left_accumulated, ds2col, DS_NAMES, data).savefig(
    f'{FIG_DIR}/fig1_distributions.pdf', bbox_inches='tight', dpi=300)

In [ ]:
build_data_fig(data, AAL_NII, ds2col, DS_NAMES).savefig(
    f'{FIG_DIR}/figdata_atlas_age.pdf', bbox_inches='tight', dpi=300)
plt.show()

Optional panel (not used in the figures above): $\langle w\rangle_k$ per decade, from the
curves computed by `precompute_parallel`. Call `plot_wk(ax, wk_calculated, norm_wk, cmap_wk, fig)`.

In [ ]:
def plot_wk(ax, wk_calculated, norm_wk, cmap_wk, fig):
    for item in wk_calculated:
        ax.loglog(item['ks'], item['w_med'], '-', lw=1.2, color=item['color'])
    ax.set_xlabel(r'$k$', fontsize=7); ax.set_ylabel(r'$\langle w \rangle_{k}$', fontsize=7)
    ax.tick_params(labelsize=6); sns.despine(ax=ax)
    sm = cm.ScalarMappable(cmap=cmap_wk, norm=norm_wk); sm.set_array([])
    cb = fig.colorbar(sm, ax=ax, fraction=0.046, pad=0.03); cb.set_label('Decade', fontsize=7)

## 4 · Figure 2 — disparity across nodes and ages

Each AAL region is assigned to a lobe, a Yeo-7 network (voxel overlap) and a strength class
(rich club / feeder / peripheral). Per region, the age–ratio correlation $r(\text{age}, R_i)$ is
computed in development and aging, and node metrics are raced against it (FDR-corrected).

**Yeo-7 atlas** (decompressed once) and the AAL90 → Yeo-7 mapping, saved to `procdata/yeo7_aal90_mapping.pkl`.

In [ ]:
YEO7_SRC = 'preprocdata/Yeo2011_7Networks_MNI152_FreeSurferConformed1mm_LiberalMask.nii.gz'
YEO7_NII = 'preprocdata/Yeo2011_7Networks_MNI152.nii'

if os.path.exists(YEO7_NII):
    print(f"{YEO7_NII} already exists")
else:
    print("Decompressing...")
    with gzip.open(YEO7_SRC, 'rb') as f_in, open(YEO7_NII, 'wb') as f_out:
        shutil.copyfileobj(f_in, f_out)
    print(f"Done: {YEO7_NII}  ({os.path.getsize(YEO7_NII)/1e6:.1f} MB)")

In [ ]:
YEO7_NAMES = {
    0: 'Other',
    1: 'Visual',
    2: 'Somatomotor',
    3: 'Dorsal Att.',
    4: 'Ventral Att.',
    5: 'Limbic',
    6: 'Fronto-Parietal',
    7: 'Default Mode',
}

aal_img  = nib.load(AAL_NII)
yeo7_img = nib.load(YEO7_NII)

# Resample Yeo7 to the AAL space (nearest neighbour)
yeo7_res  = image.resample_to_img(yeo7_img, aal_img, interpolation='nearest')
aal_data  = aal_img.get_fdata().astype(int)
yeo7_data = yeo7_res.get_fdata().astype(int)

# The AAL label values in the NIfTI are the grey levels of the readme
# (2001, 2002, ...) — the first 90 are the cortical/subcortical ROIs
aal_labels_90 = sorted([v for v in np.unique(aal_data) if v > 0])[:90]

yeo7_map = {}
for roi_idx, aal_val in enumerate(aal_labels_90, start=1):
    mask     = aal_data == aal_val
    yeo_vals = yeo7_data[mask]
    nonzero  = yeo_vals[yeo_vals > 0]
    if len(nonzero) == 0 or (yeo_vals == 0).mean() > 0.5:
        dominant = 0   # mostly subcortical / outside the cortical mantle
    else:
        counts   = np.bincount(nonzero, minlength=8)
        dominant = int(np.argmax(counts[1:]) + 1)
    yeo7_map[f'R{roi_idx}'] = YEO7_NAMES[dominant]

with open('procdata/yeo7_aal90_mapping.pkl', 'wb') as f:
    pickle.dump(yeo7_map, f)

# Check
print("Regions per network:")
print(Counter(yeo7_map.values()))
print("\nFull mapping:")
for roi, net in yeo7_map.items():
    print(f"  {roi:5s} → {net}")

**Region labels and per-subject node metrics.** `_process_subject` gives, per node, the ratio
$Y_i/Y_{null}$, degree, strength, weighted clustering (Onnela), subgraph centrality $C_{ii}$ and
total communicability $G_i$ of $e^{\tilde W}$.

In [ ]:
LOG_BINS  = np.logspace(0, np.log10(89), 50)

def sig_stars(p):
    if p < 0.001: return '***'
    if p < 0.01:  return '**'
    if p < 0.05:  return '*'
    return ''

# ---- anatomical labels ---------------------------------------------------------------
roi_names = [f'R{i+1}' for i in range(90)]

# Lobe map following the official AAL SPM12 readme
lobe_map = {}
for i in range(1, 91):
    if   1<=i<=28:   lobe_map[f'R{i}'] = 'Frontal'
    elif 29<=i<=42:  lobe_map[f'R{i}'] = 'Sub/Limbic'   # Insula+Cingulum+Hipo+Amig
    elif 43<=i<=56:  lobe_map[f'R{i}'] = 'Occipital'    # Calcarine..Fusiform
    elif 57<=i<=70:  lobe_map[f'R{i}'] = 'Parietal'
    elif 71<=i<=78:  lobe_map[f'R{i}'] = 'Sub/Limbic'   # Caudate/Putamen/Pallidum/Thalamus
    elif 79<=i<=90:  lobe_map[f'R{i}'] = 'Temporal'

# Yeo7 map from the voxel overlap
with open('procdata/yeo7_aal90_mapping.pkl', 'rb') as f:
    yeo7_map = pickle.load(f)


def _process_subject(W_raw, rho=0.99):
    W = W_raw.copy(); np.fill_diagonal(W, 0)
    k, s, Yi, kYi, _ = node_metrics(W)
    valid = k > 1

    # Nodal ratio
    Y_null = np.where(valid, 2. / (k + 1), np.nan)
    ratio  = np.where(valid, Yi / Y_null,   np.nan)

    # Clustering weighted — Onnela et al. 2005 (PRE)
    # C_i^w = 1/(k_i(k_i-1)) * sum_{j,h} (w_hat_ij * w_hat_ih * w_hat_jh)^(1/3)
    w_max  = W.max()
    W_hat  = W / w_max if w_max > 0 else W          # normalised weights ∈ [0,1]
    # cube-root geometric mean of the three weights of each triangle:
    # (W_hat^(1/3) @ W_hat^(1/3)) * W_hat^(1/3) gives the sum of the cube-root products
    W_cbrt = np.cbrt(W_hat)
    num    = np.sum((W_cbrt @ W_cbrt) * W_cbrt, axis=1)  # weighted triangles
    denom  = k * (k - 1)
    with np.errstate(invalid='ignore', divide='ignore'):
        clust_w = np.where(denom > 0, num / denom, 0.0)

    # Total communicability nodal 
    sr     = np.abs(np.linalg.eigvals(W)).max()
    A_norm = W * (rho / sr) if sr > 0 else W
    with np.errstate(over='ignore'):
        E = expm(A_norm)
    sc   = np.diag(E)        # subgraph centrality C_ii
    comm = E.sum(axis=1)     # total communicability G_i (row sums)
    return dict(ratio=ratio, k=k, s=s, clust_w=clust_w, sc=sc, comm=comm)

def build_roi_matrices(data_sub):
    conn = np.stack(list(data_sub.connectome), axis=0)
    ages = data_sub['age'].values
    Ns   = len(conn)
    print(f"  processing {Ns} subjects...")
    results = Parallel(n_jobs=N_WORKERS, prefer='threads')(
        delayed(_process_subject)(conn[i]) for i in range(Ns)
    )
    ratio_mat  = np.stack([r['ratio']   for r in results], axis=0)
    k_mat      = np.stack([r['k']       for r in results], axis=0)
    s_mat      = np.stack([r['s']       for r in results], axis=0)
    clust_mat  = np.stack([r['clust_w'] for r in results], axis=0)
    sc_mat     = np.stack([r['sc']      for r in results], axis=0)
    comm_mat   = np.stack([r['comm']    for r in results], axis=0)
    roi_means  = {
        'k':      np.nanmean(k_mat,     axis=0),
        's':      np.nanmean(s_mat,     axis=0),
        'clust_w':np.nanmean(clust_mat, axis=0),
        'sc':     np.nanmean(sc_mat,    axis=0),
        'comm':   np.nanmean(comm_mat,  axis=0),
    }
    print("  done.")
    return ratio_mat, ages, roi_means


def compute_roi_results(ratio_mat, ages, roi_means):
    N = ratio_mat.shape[1]

    roi_r = np.full(N, np.nan); roi_p = np.full(N, np.nan)
    for j in range(N):
        col = ratio_mat[:,j]; valid = ~np.isnan(col)
        if valid.sum() > 10:
            roi_r[j], roi_p[j] = pearsonr(ages[valid], col[valid])
    valid_mask = ~np.isnan(roi_p)
    _, p_fdr, _, _ = multipletests(roi_p[valid_mask], method='fdr_bh')
    roi_p_adj = np.full(N, np.nan); roi_p_adj[valid_mask] = p_fdr

    mean_s = roi_means['s']
    club   = np.where(mean_s >= np.nanpercentile(mean_s,85), 'Rich-Club',
             np.where(mean_s >= np.nanpercentile(mean_s,50), 'Feeder', 'Peripheral'))

    df_roi = pd.DataFrame({
        'ROI':   roi_names, 'r': roi_r, 'p_fdr': roi_p_adj,
        'Lobe':  [lobe_map[r] for r in roi_names],
        'Yeo7':  [yeo7_map[r] for r in roi_names],
        'Club':  club,
    }).dropna(subset=['r'])

    mean_k = roi_means['k']
    NODE_LABELS = {
        'Strength ($s_i$)':            roi_means['s'],
        'Clustering ($C^w_i$)':        roi_means['clust_w'],
        'Degree ($k_i$)':              mean_k,
        'Subgraph Centrality ($C_{ii}$)': roi_means['sc'],
        'Communicability ($G_i$)':      roi_means['comm'],
    }
    hr_rows, hr_p = [], []
    for lbl, vals in NODE_LABELS.items():
        both = ~(np.isnan(vals)|np.isnan(roi_r))
        r, p = pearsonr(vals[both], roi_r[both])
        hr_rows.append({'label': lbl, 'r': r, 'p_raw': p}); hr_p.append(p)
    _, hr_p_fdr, _, _ = multipletests(hr_p, method='fdr_bh')
    for i, row in enumerate(hr_rows):
        row['p_fdr'] = hr_p_fdr[i]; row['stars'] = sig_stars(hr_p_fdr[i])
    hr_df = (pd.DataFrame(hr_rows)
               .sort_values('r', ascending=True)
               .reset_index(drop=True))
    return df_roi, hr_df

**$kY(k)$ scatter** (Fig. 2a): 20 random subjects per dataset, log-binned mean of $kY$ per degree.

In [ ]:
def _kY_one_ds(ds, data, seed_offset=0):
    rng = np.random.default_rng(42 + seed_offset)
    sub = data[data.dataset==ds]
    sub = sub.iloc[rng.choice(len(sub), min(20,len(sub)), replace=False)]
    k_list, kY_list, acc = [], [], defaultdict(list)
    for _, row in sub.iterrows():
        k, s, Yi, kYi, _ = node_metrics(row.connectome)
        v = k > 1
        k_list.append(k[v]); kY_list.append(kYi[v])
        bin_idx = np.clip(np.digitize(k[v], LOG_BINS)-1, 0, len(LOG_BINS)-2)
        for bi in np.unique(bin_idx):
            acc[bi].extend(kYi[v][bin_idx==bi].tolist())
    return np.concatenate(k_list), np.concatenate(kY_list), dict(acc)

sc_results = Parallel(n_jobs=N_WORKERS, prefer='threads')(
    delayed(_kY_one_ds)(ds, data, i) for i, ds in enumerate(ds_list)
)
k_arr_sc  = np.concatenate([r[0] for r in sc_results])
kY_arr_sc = np.concatenate([r[1] for r in sc_results])
k_acc_sc  = defaultdict(list)
for _,_,acc in sc_results:
    for bi,vals in acc.items(): k_acc_sc[bi].extend(vals)
bin_centers  = np.sqrt(LOG_BINS[:-1]*LOG_BINS[1:])
valid_bins   = [bi for bi in sorted(k_acc_sc) if len(k_acc_sc[bi])>=3]
ks_obs_sc    = bin_centers[valid_bins]
kY_obs_mu_sc = np.array([np.mean(k_acc_sc[bi]) for bi in valid_bins])

**Development and aging.** Per-region age–ratio correlations and the node-metric race, for age
$\le$ `AGE_SPLIT` and age $\ge$ `AGE_SPLIT`; everything the figures need is saved to
`procdata/fig2_precomp.pkl`.

In [ ]:
print(f"=== DEVELOPMENT (age <= {AGE_SPLIT}) ===")
data_dev = data[data.age <= AGE_SPLIT].reset_index(drop=True)
R_dev, ages_dev, means_dev = build_roi_matrices(data_dev)
df_dev, hr_dev = compute_roi_results(R_dev, ages_dev, means_dev)

print(f"=== AGING (age >= {AGE_SPLIT}) ===")
data_age = data[data.age >= AGE_SPLIT].reset_index(drop=True)
R_age, ages_age, means_age = build_roi_matrices(data_age)
df_age, hr_age = compute_roi_results(R_age, ages_age, means_age)

with open('procdata/fig2_precomp.pkl', 'wb') as f:
    pickle.dump(dict(
        df_dev=df_dev, hr_dev=hr_dev,
        df_age=df_age, hr_age=hr_age,
        k_arr_sc=k_arr_sc, kY_arr_sc=kY_arr_sc,
        ks_obs_sc=ks_obs_sc, kY_obs_mu_sc=kY_obs_mu_sc,
    ), f)
print("\nsaved procdata/fig2_precomp.pkl")
print("\n--- Horse-race DEV ---"); print(hr_dev[['label','r','stars']].to_string())
print("\n--- Horse-race AGE ---"); print(hr_age[['label','r','stars']].to_string())

**Figure style** for the Figure 2 panels (palettes, box/strip styles, significance stars). Re-loads `fig2_precomp.pkl`, so the figures can be redrawn without recomputing.

In [ ]:
with open('procdata/fig2_precomp.pkl', 'rb') as f:
    _d = pickle.load(f)
df_dev       = _d['df_dev'];      hr_dev       = _d['hr_dev']
df_age       = _d['df_age'];      hr_age       = _d['hr_age']
k_arr_sc     = _d['k_arr_sc'];    kY_arr_sc    = _d['kY_arr_sc']
ks_obs_sc    = _d['ks_obs_sc'];   kY_obs_mu_sc = _d['kY_obs_mu_sc']

sns.set_theme(style='ticks', context='paper')

# ---- palettes ------------------------------------------------------------------------
LOBE_ORDER = ['Temporal','Sub/Limbic','Parietal','Occipital','Frontal']
CLUB_ORDER = ['Rich-Club','Feeder','Peripheral']
LOBE_PAL   = dict(zip(LOBE_ORDER, sns.color_palette('muted', len(LOBE_ORDER))))
YEO_NETS   = sorted(df_dev['Yeo7'].unique())
YEO_ORDER  = YEO_NETS
YEO_PAL    = dict(zip(YEO_ORDER, sns.color_palette('tab10', len(YEO_ORDER))))
CLUB_PAL   = {'Rich-Club':'#e74c3c','Feeder':'#f39c12','Peripheral':'#3498db'}
BOX_KW     = dict(showfliers=False, linewidth=0.7, zorder=2, width=0.5)
STRIP_KW   = dict(color='#222', alpha=0.5, size=2.5, zorder=3)


def add_stars(ax, df, group_col, order):
    ylim  = ax.get_ylim()
    ystar = ylim[1] - 0.02*(ylim[1]-ylim[0])
    for xi, grp in enumerate(order):
        pvals = df.loc[df[group_col]==grp, 'p_fdr'].dropna()
        if len(pvals) == 0: continue
        s = sig_stars(pvals.median())
        if s: ax.text(xi, ystar, s, ha='center', va='bottom',
                      fontsize=6.5, fontweight='bold')

In [ ]:
# =============================================================================
# fig2_grid: kY scatter + trajectories of R, <C_ii> and <C_w>
# =============================================================================
fig1 = plt.figure(figsize=(3.4,4.5))
gs1 = gridspec.GridSpec(2, 2, figure=fig1,wspace=0.65, hspace=0.5)

ax_kY    = fig1.add_subplot(gs1[0, 0])
ax_tdis  = fig1.add_subplot(gs1[0, 1])
ax_tcomm = fig1.add_subplot(gs1[1, 0])
ax_tclus = fig1.add_subplot(gs1[1, 1])

# --- Panel (a): kY scatter ---
rng_sc = np.random.default_rng(0)
idx_sc = rng_sc.choice(len(k_arr_sc), min(7000, len(k_arr_sc)), replace=False)

ax_kY.scatter(k_arr_sc[idx_sc], kY_arr_sc[idx_sc],
              s=0.4, alpha=0.07, color='steelblue', rasterized=True, zorder=1)
ax_kY.fill_between(NUL['ks'], NUL['kY_lo'], NUL['kY_hi'],
                   color='grey', alpha=0.25, zorder=2)

lr = linregress(np.log10(ks_obs_sc), np.log10(kY_obs_mu_sc))
ax_kY.text(0.95, 0.05, rf'$\Upsilon\sim k^{{{lr.slope:.2f}}}$',
           transform=ax_kY.transAxes, ha='right', va='bottom', fontsize=6)

ax_kY.plot(NUL['ks'], NUL['kY_null'], '-',  color='grey',    lw=1,   label='null ±2σ')
ax_kY.plot(NUL['ks'], NUL['kY_het'],  '--', color='#cc0000', lw=0.9, label='heterog.')
ax_kY.plot(ks_obs_sc, kY_obs_mu_sc,   '-',  color='steelblue', lw=1.6, label=r'$\langle kY\rangle_k$')

ax_kY.set_xscale('log'); ax_kY.set_yscale('log')
ax_kY.set_xlim(2, 61);   ax_kY.set_ylim(1, 20)
ax_kY.set_xlabel('$k$'); ax_kY.set_ylabel(r'$kY(k)$')
ax_kY.legend(fontsize=5, handlelength=1, loc='upper left', frameon=True, framealpha=0.9)
sns.despine(ax=ax_kY)

# --- Panels (b, c, d): Trajectories ---
traj_cfg = [
    (ax_tdis,  'Ratio',  r'$\langle Y/Y_{null}\rangle$', 1.2, 3),
    (ax_tcomm, 'sc_mean', r'$\langle C_{ii}\rangle$',  0.9, 2),
    (ax_tclus, 'C_w',       r'$\langle C_w\rangle$',      0.9, 2),
]

for ax, metric, ylabel, lw, ms in traj_cfg:
    for ds in ds_list:
        sub = data[data.dataset==ds].sort_values('age')
        if metric not in sub.columns or len(sub)==0: continue
        g = sub.groupby('age')[metric].mean()
        ax.plot(g.index, g.values, 'o-', ms=ms, lw=lw, color=ds2col[ds], alpha=0.85)
    ax.set_ylabel(ylabel)
    ax.set_xlabel('Age')
    sns.despine(ax=ax)


for ax, lab in zip([ax_kY, ax_tdis, ax_tcomm, ax_tclus], 'abcd'):
    ax.text(-0.15, 1.1, lab, transform=ax.transAxes,
            fontsize=9, fontweight='bold', va='top', ha='right')

fig1.savefig(f'{FIG_DIR}/fig2_grid.pdf', bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
# =============================================================================
# fig2_boxplots: per-region age–ratio r by lobe, Yeo-7 network and strength class
# =============================================================================
fig2, axes2 = plt.subplots(3, 1, figsize=(3.4, 4.5), gridspec_kw={'hspace': 0.8})

for ax, col, order, pal in [
    (axes2[0], 'Lobe', LOBE_ORDER, LOBE_PAL),
    (axes2[1], 'Yeo7', YEO_ORDER,  YEO_PAL),
    (axes2[2], 'Club', CLUB_ORDER, CLUB_PAL)
]:
    sns.boxplot(data=df_dev, x=col, y='r', order=order,
                palette=pal, hue=col, legend=False, ax=ax, **BOX_KW)
    sns.stripplot(data=df_dev, x=col, y='r', order=order,
                  ax=ax, **STRIP_KW)
    
    ax.axhline(0, ls='--', lw=0.6, color='grey', zorder=1)
    ax.set_xlabel('')
    ax.set_xticks(range(len(order)))
    ax.set_xticklabels(order, rotation=35, ha='right', fontsize=6)
    add_stars(ax, df_dev, col, order)
    sns.despine(ax=ax)

axes2[0].set_ylabel(r'Age–Ratio $r$')
axes2[1].set_ylabel(r'Age–Ratio $r$')
axes2[2].set_ylabel(r'Age–Ratio $r$')

# Synchronize ylim
ymin = min(a.get_ylim()[0] for a in axes2)
ymax = max(a.get_ylim()[1] for a in axes2)
for a in axes2: 
    a.set_ylim(ymin, ymax)

# Re-apply stars with synced limits
for ax, col, order in zip(axes2, ['Lobe', 'Yeo7', 'Club'], [LOBE_ORDER, YEO_ORDER, CLUB_ORDER]):
    add_stars(ax, df_dev, col, order)

fig2.savefig(f'{FIG_DIR}/fig2_boxplots.pdf', bbox_inches='tight', dpi=300)
plt.show()

Node-metric race in development as a LaTeX table:

In [ ]:
tbl = hr_dev[['label','r','p_fdr']].copy()
tbl['r']     = tbl['r'].map('{:.3f}'.format)
tbl['p_fdr'] = tbl['p_fdr'].map(lambda p: f'{p:.1e}' if p < 1e-3 else f'{p:.3f}')
tbl.columns  = ['Node Metric', '$r$', '$p_{\\mathrm{FDR}}$']
print(tbl.to_latex(index=False, escape=False, column_format='lcc'))

Weighted clustering against age in aging, raw and controlling for density:

In [ ]:
def partial_corr(x, y, z):
    """corr(x, y | z) by linear residualisation."""
    rx = x - np.polyval(np.polyfit(z, x, 1), z)
    ry = y - np.polyval(np.polyfit(z, y, 1), z)
    return pearsonr(rx, ry)

# per subject (aging cohort, where C_w rises)
sub   = data[data.age >= AGE_SPLIT]
conn  = np.stack(list(sub.connectome))
ages  = sub['age'].values
dens  = np.array([(W > 0).sum() / (90*89) for W in conn])          # graph density
cw    = np.array([np.nanmean(_process_subject(W)['clust_w']) for W in conn])

r_raw,  p_raw  = pearsonr(ages, cw)
r_part, p_part = partial_corr(cw, ages, dens)
print(f"C_w–age  raw r={r_raw:.3f} (p={p_raw:.1e})")
print(f"C_w–age | density  r={r_part:.3f} (p={p_part:.1e})")

## 5 · Supplementary

Per-region age–ratio correlations in development by lobe, Yeo-7 network and strength class
(`figSI_Y_horserace.pdf`).

In [ ]:
fig_si = plt.figure(figsize=(7.0,2.5))
gs = gridspec.GridSpec(1, 3, figure=fig_si, wspace=0.65, hspace=0.75)
ax_e, ax_f = fig_si.add_subplot(gs[0,0]), fig_si.add_subplot(gs[0,1])
ax_g = fig_si.add_subplot(gs[0,2])

for ax, col, order, pal, rot in [
    (ax_e, 'Lobe', LOBE_ORDER, LOBE_PAL, 35),
    (ax_f, 'Yeo7', YEO_ORDER,  YEO_PAL,  40),
    (ax_g, 'Club', CLUB_ORDER, CLUB_PAL, 25),
]:
    sns.boxplot(data=df_dev, x=col, y='r', order=order,
                palette=pal, hue=col, legend=False, ax=ax, **BOX_KW)
    sns.stripplot(data=df_dev, x=col, y='r', order=order, ax=ax, **STRIP_KW)
    ax.axhline(0, ls='--', lw=0.6, color='grey', zorder=1)
    ax.set_xlabel('')
    ax.set_xticks(range(len(order)))
    ax.set_xticklabels(order, rotation=rot, ha='right',
                       fontsize=6)
    ax.set_ylabel(r'Age–$R$  $r$ (per ROI)')
    sns.despine(ax=ax)

ymin = min(a.get_ylim()[0] for a in (ax_e, ax_f, ax_g))
ymax = max(a.get_ylim()[1] for a in (ax_e, ax_f, ax_g))
for a in (ax_e, ax_f, ax_g): a.set_ylim(ymin, ymax)
for ax, col, order in [(ax_e,'Lobe',LOBE_ORDER),(ax_f,'Yeo7',YEO_ORDER),(ax_g,'Club',CLUB_ORDER)]:
    add_stars(ax, df_dev, col, order)


for ax, lab in zip([ax_e, ax_f, ax_g], 'abc'):
    ax.text(-0.18, 1.08, lab, transform=ax.transAxes,
            fontsize=9, fontweight='bold', va='top', ha='right')

fig_si.savefig(f'{FIG_DIR}/figSI_Y_horserace.pdf', bbox_inches='tight', dpi=300)
plt.show()